# Line Model — Byte-Level BPE + RoPE

Encoder-decoder for line completion. Two architectural upgrades:
* **Byte-level BPE tokenizer** (shared with the Token model).
* **RoPE inside *self-attention*** (encoder + decoder). Cross-attention stays standard — Q (decoder positions) and K (encoder positions) live in different position spaces, so rotating both with the same RoPE doesn't make sense.

Custom encoder-decoder blocks because `nn.Transformer` doesn't expose Q/K for rotation. Uses `F.scaled_dot_product_attention` for the efficient masked path (causal, padding, cross). Keeps warmup+cosine, label smoothing, mixed precision, and early stopping.

In [12]:
%tb
import os, json, math, random, glob, tempfile
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

from tokenizers import Tokenizer
from tokenizers.implementations import ByteLevelBPETokenizer

import warnings
warnings.filterwarnings("ignore")

from modules.plotting import MetricLog, plot_metrics
from modules.early_stopping import EarlyStopping
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.datasets.loading import *
from modules.datasets.line_dataset import *
from modules.tokenizers.BPE_tokenizer import *
from modules.models.L_rope_model import *

# WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
WORKDIR = r'C:\Programing\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")
LINE_MODEL_NAME = 'line_model_bpe_rope'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

KeyboardInterrupt: 

WORKDIR: C:\Programing\code_autocomplete
[Device] cuda


## Training loop

In [13]:
def _clip_norm(model, max_norm=1.0):
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(model, train_dl, val_dl, epochs, lr, device,
                     saver, log, plot_dir,
                     label_smoothing=0.1, warmup_frac=0.05,
                     patience=3, use_amp=True):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], label_smoothing=label_smoothing)
    amp_enabled = use_amp and device.type == "cuda"
    scaler = GradScaler("cuda", enabled=amp_enabled)
    stopper = EarlyStopping(patience=patience)
    PAD = SPECIAL["<PAD>"]

    for ep in range(1, epochs + 1):
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0; gn = 0.0
        for src, tgt in tqdm(train_dl, desc=f"[Line] Epoch {ep}/{epochs} train",
                             leave=False, unit="batch"):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            src_pad = (src == PAD)
            dec_in = tgt[:, :-1]
            dec_out = tgt[:,  1:]
            tgt_pad = (dec_in == PAD)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(src, dec_in,
                               src_key_padding_mask=src_pad,
                               tgt_key_padding_mask=tgt_pad)
                loss = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt); scaler.update(); sched.step()
            t_loss += loss.item()
            t_steps += 1
        tl = t_loss / t_steps

        model.eval()
        v_loss = v_steps = v_acc = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                 leave=False, unit="batch"):
                src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
                src_pad = (src == PAD)
                dec_in = tgt[:, :-1]; dec_out = tgt[:, 1:]
                tgt_pad = (dec_in == PAD)
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    logits = model(src, dec_in,
                                   src_key_padding_mask=src_pad,
                                   tgt_key_padding_mask=tgt_pad)
                    loss = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                preds = logits.argmax(-1)
                mask = (dec_out != PAD)
                if mask.any():
                    v_acc += (preds[mask] == dec_out[mask]).float().mean().item()
                v_loss += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        va = v_acc / v_steps if v_steps else 0.0

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"], token_acc=va, grad_norm=gn)
        tqdm.write(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
                   f"  ppl={math.exp(min(vl,20)):.1f}  acc={va:.3f}"
                   f"  lr={opt.param_groups[0]['lr']:.2e}")
        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break
    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [15]:
class Arguments():
    def __init__(self,
                 data_dir=f"{WORKDIR}/Clean_Dataset",
                 ckpt_dir=f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                 plot_dir=f"{WORKDIR}/plots/{LINE_MODEL_NAME}",
                 tokenizer=f"tokenizer_bpe.json",
                 epochs=5, batch=32, lr=5e-4, ctx=128,
                 d_model=256, n_layers=4, n_heads=8,
                 vocab_size=16000, max_files=0, val_split=0.1, seed=42,
                 label_smoothing=0.1, warmup_frac=0.05, patience=3, use_amp=True,
                 skip_line=False, test=False):
        for k, v in locals().items():
            if k != "self": setattr(self, k, v)


def main():
    args = Arguments()
    # args = Arguments(max_files=100, epochs=2)
    # args = Arguments(test=True)

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(args.seed)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)

    # ── BPE tokenizer (re-use the file the Token model trained, if it exists) ─
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = BPECodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] training byte-level BPE from data …")
        texts = load_files(args.data_dir, args.max_files)
        if not texts: print("[ERROR] no data files found."); return
        tokenizer = BPECodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)
        print(f"[Tokenizer] saved to {args.tokenizer}")

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads, n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )
    torch.serialization.add_safe_globals([ModelCfg])

    if args.test:
        lm = LineModel(cfg).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{LINE_MODEL_NAME}_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
        hand_test_repl(None, lm, tokenizer, None, device); return

    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts: print("[ERROR] no data files found."); return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt, va_txt = texts[:split], texts[split:]

    if not args.skip_line:
        print("  Preparing LINE model (BPE + RoPE)")
        tr_ds = LineDataset(tr_txt, tokenizer)
        va_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,
                           collate_fn=collate, num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False,
                           collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters (BPE+RoPE)")

        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log = MetricLog()
        print("  Training LINE model")
        train_line_model(
            model=line_model, train_dl=tr_dl, val_dl=va_dl,
            epochs=args.epochs, lr=args.lr, device=device,
            saver=line_saver, log=line_log, plot_dir=args.plot_dir,
            label_smoothing=args.label_smoothing, warmup_frac=args.warmup_frac,
            patience=args.patience, use_amp=args.use_amp,
        )
        # hand_test_repl(None, line_model, tokenizer, None, device)


main()

[Tokenizer] loading tokenizer_bpe.json
[Loading] Started loading
[Data] loaded 12106 files from C:\Programing\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Preparing LINE model (BPE + RoPE)
[LineDataset] 869019 samples
[LineDataset] 94968 samples
[Line  Model] 15.55M parameters (BPE+RoPE)
  Training LINE model
[Line] DataLoader — 27157 train batches, 2968 val batches
Epoch 1


[Line  ep   1] train_loss=1.6132  val_loss=1.3777  ppl=4.0  acc=0.998  lr=4.70e-04
[Saver] saved ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep001_loss1.3777.pt  (val_loss=1.3777)
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_ep01.png
Epoch 2


[Line  ep   2] train_loss=1.3318  val_loss=1.3703  ppl=3.9  acc=0.998  lr=3.50e-04
[Saver] saved ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep002_loss1.3703.pt  (val_loss=1.3703)
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_ep02.png
Epoch 3


[Line  ep   3] train_loss=1.3268  val_loss=1.3620  ppl=3.9  acc=0.998  lr=1.89e-04
[Saver] saved ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep003_loss1.3620.pt  (val_loss=1.3620)
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_ep03.png
Epoch 4


[Line  ep   4] train_loss=1.3225  val_loss=1.3522  ppl=3.9  acc=0.998  lr=5.27e-05
[Saver] removed old ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep001_loss1.3777.pt
[Saver] saved ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep004_loss1.3522.pt  (val_loss=1.3522)
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_ep04.png
Epoch 5


[Line  ep   5] train_loss=1.3200  val_loss=1.3512  ppl=3.9  acc=0.998  lr=0.00e+00
[Saver] removed old ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep002_loss1.3703.pt
[Saver] saved ckpt: C:\Programing\code_autocomplete\checkpoints\line_model_bpe_rope\line_model_bpe_rope_ep005_loss1.3512.pt  (val_loss=1.3512)
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_ep05.png
[Plot] saved → C:\Programing\code_autocomplete/plots/line_model_bpe_rope/line_model_bpe_rope_final.png
